In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/term-deposit-marketing-2020.csv")

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numerical_cols = [
    "age",
    "balance",
    "day",
    #"duration",
    "campaign"
]

categorical_cols = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month"
]

"""
preprocessor = ColumnTransformer(
    transformers=[
        ("number", StandardScaler(), numerical_cols),
        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)"""




'\npreprocessor = ColumnTransformer(\n    transformers=[\n        ("number", StandardScaler(), numerical_cols),\n        ("category", OneHotEncoder(handle_unknown="ignore"), categorical_cols)\n    ]\n)'

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["y", "duration"])
y = df["y"].map({"no": 0, "yes": 1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
#feature selection of the original features (not the one-hot encoded features)

import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureSelector(BaseEstimator, TransformerMixin):

    def __init__(self, k="all"):
        self.k = k

    def fit(self, X, y):

        # don't modify original data
        X = X.copy()
        X_encoded = X.copy()

        for column in categorical_cols:
            X_encoded[column] = X_encoded[column].astype("category").cat.codes

        # Calculate feature scores
        self.scores_ = mutual_info_classif(X_encoded, y, random_state=42)

        # Rank features from highest to lowest score
        self.indices_ = np.argsort(self.scores_)[::-1]

        if self.k == "all":
            self.selected_indices_ = self.indices_
        else:
            self.selected_indices_ = self.indices_[:self.k]

        #change selected features to original feature names
        self.selected_features_ = X.columns[self.selected_indices_].tolist()

        return self

    #return dataset with only the selected features
    def transform(self, X):
        return X[self.selected_features_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.selected_features_)

In [6]:
class DynamicPreprocessor(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.preprocessor_ = None

    def fit(self, X, y=None):

        # Select only the features that are present in the dataset
        selected_numerical = [col for col in numerical_cols if col in X.columns]
        selected_categorical = [col for col in categorical_cols if col in X.columns]

        self.preprocessor_ = ColumnTransformer(
            transformers=[
                ("number", StandardScaler(), selected_numerical),
                ("category", OneHotEncoder(handle_unknown="ignore"), selected_categorical)
            ]
        )

        self.preprocessor_.fit(X, y)

        return self

    # return the transformed dataset with only the selected features
    def transform(self, X):
        return self.preprocessor_.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.preprocessor_.get_feature_names_out(input_features)

"Random Forest": {
        "pipeline": Pipeline([
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(random_state=42, class_weight="balanced"))
        ]),
        "params": {
            "model__n_estimators": [50, 100, 150, 200],
            "model__max_depth": [1, 2, 5, 10],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
    },

In [ ]:
#setup baseline models

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier

"""try:
 - svm
 - knn
 - random forest
 - xgboost"""

#variables for models

#feature selector
feature_selector = FeatureSelector()

#preprocessor
preprocessor = DynamicPreprocessor()

#scale pos weight for xgboost
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]


models = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("select", feature_selector),
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"))
        ]),
        "params": {
            "model__C": [0.01, 0.1, 1, 10],
            "select__k": [3, 5, 7, 10, "all"]
        }
    }, 

    "XGBoost": {
            "pipeline": Pipeline([
                ("select", feature_selector),
                ("preprocessor", preprocessor),
                ("model", XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight))
            ]),
            "params": {
                "model__max_depth": [1, 2, 3, 4, 6, 10],
                "model__learning_rate": [0.05, 0.1, 0.2],
                "select__k": [3, 5, 7, 10, "all"]
            }
        },

    

        "KNN": {
        "pipeline": Pipeline([
            ("select", feature_selector),
            ("preprocessor", preprocessor),
            ("model", KNeighborsClassifier())
        ]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9, 11],
            "model__weights": ["uniform", "distance"],
            "select__k": [3, 5, 7, 10, "all"]
        }
    },


    #"SVM": {
    #    "pipeline": Pipeline([
    #        ("select", feature_selector),
    #        ("preprocessor", preprocessor),
    #        ("model", SVC(random_state=42, class_weight="balanced", probability=True))
    #    ]),
    #    "params": {
    #        "model__C": [0.01, 0.1, 1, 10],
    #        "select__k": [3, 5, 7, 10, "all"]
    #    }
    }
}

In [8]:
from sklearn.model_selection import StratifiedKFold


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
import time
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_validate


results = []

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0)
}


for name, config in models.items():
    print(f"Training model: {name}")

    # Start timer
    start_time = time.perf_counter()

    # Grid search
    grid_search = GridSearchCV(config["pipeline"], config["params"], cv=cv, scoring="f1", n_jobs=-1)

    grid_search.fit(X_train, y_train)

    # Get best model
    best_model = grid_search.best_estimator_

    # Get feature selection information
    selector = best_model.named_steps["select"]

    selected_features = selector.selected_features_
    feature_scores = selector.scores_

    # Evaluate best model
    cv_results = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    
    elapsed_time = time.perf_counter() - start_time

    # Record results
    results.append({
        "Model": name,
        "Accuracy": cv_results["test_accuracy"].mean(),
        "Precision": cv_results["test_precision"].mean(),
        "Recall": cv_results["test_recall"].mean(),
        "F1 Score": cv_results["test_f1"].mean(),
        "Training Time (seconds)": elapsed_time,
        "Selected Features": selected_features,
        "Feature Scores": feature_scores,
        "Best Parameters": grid_search.best_params_
    })


Training model: Logistic Regression
Training model: XGBoost
Training model: KNN
Training model: SVM


c:\Users\Shane\Desktop\Other\vscode\Github Projects\04AWIXoLex7K20GE\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [10]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

              Model  Accuracy  Precision   Recall  F1 Score  Training Time (seconds)                                                                              Selected Features                                                                                                                                                                                                                                              Feature Scores                                                         Best Parameters
Logistic Regression  0.657656   0.121934 0.601200  0.202746                19.387307 [month, housing, contact, age, day, balance, marital, education, job, campaign, default, loan] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353]                                   {'model__C': 0.1, 'select__k':

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Train final model on training data
final_model = best_model
final_model.fit(X_train, y_train)

# Final test prediction
y_pred = final_model.predict(X_test)

# Final test metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("Final Test Results")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("\nConfusion Matrix:")
print(cm)

try dropping duration to see the precision, recall, and f1 results

try:
 - svm
 - knn
 - random forest
 - xgboost

In [ ]:
"""

Model               Accuracy  Precision   Recall  F1 Score  Training Time (seconds)                                                                              Selected Features                                                                                                                                                                                                                                              Feature Scores                                                         Best Parameters
Logistic Regression  0.657656   0.121934 0.601200  0.202746                19.387307 [month, housing, contact, age, day, balance, marital, education, job, campaign, default, loan] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353]                                   {'model__C': 0.1, 'select__k': 'all'}
            XGBoost  0.846094   0.188685 0.341812  0.243071                48.372505                [month, housing, contact, age, day, balance, marital, education, job, campaign] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353]  {'model__learning_rate': 0.1, 'model__max_depth': 10, 'select__k': 10}
                KNN  0.901375   0.214903 0.129472  0.159716               120.057894                                                            [month, housing, contact, age, day] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353] {'model__n_neighbors': 3, 'model__weights': 'distance', 'select__k': 5}
                SVM  0.736313   0.143635 0.532141  0.226126              3045.730321 [month, housing, contact, age, day, balance, marital, education, job, campaign, default, loan] [0.006338099325582158, 0.0025700594961286516, 0.0035518988493001835, 0.002606710993709349, 2.2253229923885343e-05, 0.004185981395054217, 0.008903672379657612, 0.0, 0.00683381606086142, 0.006323285113879251, 0.012808924633678664, 0.0015177271233184353]                                     {'model__C': 1, 'select__k': 'all'}



Note:

SVM consistently takes a much longer time to train than the other models, and does not show significant improvement.
It is for this reason that I have commented it out of the models dictionary. 

"""